### Import libraries and also define variables

In [ ]:
import fennec_ml as fn
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import json
from torch.utils.data import TensorDataset, DataLoader
import glob
import re
import matplotlib.pyplot as plt

#Check Pytorch Version
torch.__version__

'2.14.0+cpu'

In [ ]:
# --- FOLDER CONFIG ---
data_dir = os.path.join(os.getcwd(), "testing_data")
os.makedirs(data_dir, exist_ok=True)

#Make sure these are correct
excel_dir = data_dir = r"C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Raw_data"
csv_dir = r"C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Processed_data"

# --- MAKE VARS OF INTEREST ---
base_dir = os.getcwd()
json_path = os.path.join(base_dir, "3D_vars_of_interest.json")

# Data to write - add or remove as needed
vars_of_interest = {
    "IMU_0": ['GyrX', 'GyrY', 'GyrZ', 'AccX', 'AccY', 'AccZ'],
    #"RCOU": ['C1','C2','C3','C4'],
    #"ATT": ["Roll","Pitch","Yaw"],
    #"ATT": ["DesRoll","Roll","DesPitch","Pitch","DesYaw","Yaw","ErrRP","ErrYaw"],
    #"RATE": ["RDes","R","ROut","PDes","P","POut","YDes","Y","YOut"], # Not sure but could have been 88% acuracy
    #"VIBE_0": ["VibeX", "VibeY", "VibeZ"],
    #"VIBE_1": ["VibeX", "VibeY", "VibeZ"]
}

# Write to JSON file
with open(json_path, "w") as f:
    json.dump(vars_of_interest, f, indent=4)

print(f"Created {json_path}")

Created c:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 2-code simple model\3D_vars_of_interest.json


In [ ]:
# --- EXCEL TO CSV ---
#Remember to clear out the old data before writing more to the processed data
print("Using folder_cleaner() to convert from raw excel files to useful .csv's.\nfolder_cleaner() output:")

print("Running folder_cleaner...")
fn.folder_cleaner(excel_dir, csv_dir, overwrite= True, varspath="3D_vars_of_interest.json", downsample=False)
print("folder_cleaner finished")

Using folder_cleaner() to convert from raw excel files to useful .csv's.
folder_cleaner() output:
Running folder_cleaner...
289G_0.15A_L_1.xlsx processed and saved to C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Processed_data as 289G_0.15A_L_1.csv
folder_cleaner finished


How should I define weight(m) and bias(b)? Start and end and step? nn.linear solves this.

### Y = mx + b

Does X need to be flight data and the Y need to be the CG location?? How do I organize this.

This output is worng techically because it displays each data point when in reality it will take one input(Flight data) and output one output(CG guess)

It needs to have weights and biases for the X data that give the proper Y value(CG location)

In [ ]:
#this is forging a fake data set for testing
#Create *known* parameters 
weight = 0.9 #m (slope)
bias = 0.3 #b (intercept)

# How should i define start and end and step would it be the limits for CG and the interval of precision we want to measure at
# Such as start = -0.75 (inches), end = 0.75 (inches) with a step of 0.15 #This is the CG movment 
start = -0.75
end = 0.751 # added the 0.001 to include th evalue of 0.75
step = 0.15
# X is from 0 to 1 witha step of 0.02
X = torch.arange(start, end, step).unsqueeze(dim = 1) #Unsqueeze adds extra dimensions, so bsically each square bracket around each number to sperate the data better
y = weight*X + bias #Linear Regression Equation

#Given X flight data it spits out y CG location

#Display x and y values for the first 11 elements
X[:11],y[:11] #11 total postions including the datum, these are the correct positions of the CG
#len(X)


(tensor([[-0.7500],
         [-0.6000],
         [-0.4500],
         [-0.3000],
         [-0.1500],
         [ 0.0000],
         [ 0.1500],
         [ 0.3000],
         [ 0.4500],
         [ 0.6000],
         [ 0.7500]]),
 tensor([[-0.3750],
         [-0.2400],
         [-0.1050],
         [ 0.0300],
         [ 0.1650],
         [ 0.3000],
         [ 0.4350],
         [ 0.5700],
         [ 0.7050],
         [ 0.8400],
         [ 0.9750]]))

Basically we  want the model to be able to tune the weights and biases to be able to get the same y values from the above cell based off the given flight data.

Model: Trying to make one guess per flight log, 
So it needs a confidence percentage that it believes is the likely hood of each CG location based off the data

Now how do we ge the flight characteristics to a CG guess? 
1. what data do we need from the flight controller
2. How do I organize that?

In [9]:
# Create train/test split
#The int makes sure it is an integer cannot have 0.5 of a data in one of the splits
train_split = int(0.8 * len(X)) # 80 percent of the total X values become the train split

# The train section takes variables from 0 to train_split
X_train, y_train = X[:train_split], y[:train_split]
# The test section takes variables from train_split to the end
X_test, y_test = X[train_split:], y[train_split:]
#They are like this so the data does not overlap when splitting the data

len(X_train), len(y_train), len(X_test), len(y_test)

(8, 8, 3, 3)

In [10]:
def plot_predictions(train_data=X_train,
                     train_labels=y_train,
                     test_data=X_test,
                     test_labels=y_test,
                     predictions=None):
    """
    Plots training data, test data and comapres predicitons.
    """
    plt.figure(figsize=(10, 7))

    # Plot training data
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")

    #Plot tests data in green
    plt.scatter(test_data, test_labels, c="g", s=4, label="Testing data")

    # Are there predictions?
    if predictions is not None:
        #Plot the predictions if they exist
        plt.scatter(test_data, predictions, c="r", s=4, label = "Predictions")

        #Show legend
    plt.legend(prop={"size":14});

### This is the Model

How can I make the parameters better than rand?

may need to add a hidden layer ReLU if this plateaus


I thin kthis is too bare bones nn.linear may need to be used but im not sure yet

In [11]:
# Create a linear regression model class
class LinearRegressionModel(nn.Module): # almost everything in Pytorch inherits it from nn.Module
    def __init__(self):
        super().__init__()
        #This sets intial conditions for weights and bias in the equation
        self.weights = nn.Parameter(torch.randn(1,
                                               requires_grad=True,
                                               dtype=torch.float))
        self.bias = nn.Parameter(torch.randn(1,
                                               requires_grad=True,
                                               dtype=torch.float))
    #Forward method to define the computation in the model
    def forward(self, x: torch.Tensor) -> torch.Tensor: #x is the input data
        return self.weights * x + self.bias #linear Regression Model


Maybe use this?

In [ ]:
class FlightCGModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=7, padding=3), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=7, padding=3), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)   # collapses time -> 1 value per channel
        self.head = nn.Linear(64, 1)

    def forward(self, x):                      # x: (batch, 6, time)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)           # (batch, 64)
        return self.head(x)  

In [12]:
#definign vars for Model 2
num_features = 1

In [13]:
# Create a linear regression model class
class LinearRegressionModel2(nn.Module): # almost everything in Pytorch inherits it from nn.Module
    def __init__(self):
        super().__init__()
        self.linear_layer = nn.Linear(in_features=num_features, out_features=1)
        
    #Forward method to define the computation in the model
    def forward(self, x: torch.Tensor) -> torch.Tensor: #x is the input data
        return self.linear_layer(x)

The random seed only applies to the random parameters

In [14]:
#Create random seed for reproducibility
torch.manual_seed(42) #sets intial conditions biscally, create repeatibility

#Create instance of the model (this is a subclass of nn.module)
LRModel = LinearRegressionModel()

#CHeck out parameters
list(LRModel.parameters())

[Parameter containing:
 tensor([0.3367], requires_grad=True),
 Parameter containing:
 tensor([0.1288], requires_grad=True)]

In [15]:
# Make predictiosn with model
with torch.inference_mode(): #This is quicker
    y_preds = LRModel(X_test)

#Can use this too but inference mode is favored
# with torch.no_grad():
#     y_preds = model_0(X_test)

y_preds;

#NOTimplementederror: probably means an indetation is wrong in the model section

### Optimizer

I think I will use an Adam or Adamw optimizer because it cahgnes the learnign rate as we appraoch the minimun when performing gradient desecnt 

This create a quicker and more efficient trining time without over stepping the min

In [16]:
# Set up a loss function
loss_fn = nn.L1Loss()

# Setup an optimizer (Adam or SGD  is built optimizer)# Stocahstic gradient descent

optimizer = torch.optim.SGD(params = LRModel.parameters(),
                            lr = 0.01) #lr = learning rate, possibly the most important hyperparameter

### Training loop

In [17]:
# An epoch is one loop through the data
epochs = 1 #Hyperparameter

### Training
# 0. Loop through data
for epochs in range(epochs):
    # Set the model to training mode
    LRModel.train() #train mode in Pytorch sets all parameters that require gradients to require gradients

    #1. Forward pass
    y_pred = LRModel(X_train)

    # 2. Calculate the loss
    loss = loss_fn(y_pred,y_train)

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4.Backward propagation on the loss with respect to the parameters of the model
    loss.backward()

    # 5. Step the optimizer (perform gradient descent)
    optimizer.step() #by default how the optimizer cahnges will accumulate through the loop... we have to zero them above in step 3

    ###Testing
    LRModel.eval() #turns off gradient tracking
